In [ ]:
# ════════════════════════════════════════════════════════════════
# M3_F05 — ALCHEMIST  |  Cellule 1 : Drive + Install
# Prérequis : GPU T4 activé (Exécution > Modifier le type d'exécution)
# ════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

!pip install -q flask flask-cors
!pip install -q playwright
!playwright install chromium
!playwright install-deps chromium

import os, sys
from pathlib import Path

CODEBASE = Path("/content/drive/MyDrive/EXODUS_V2/03_MODE_ASCENSION/F05_ALCHEMIST/CODEBASE")
sys.path.insert(0, str(CODEBASE))
print("Drive monté — CODEBASE :", CODEBASE)

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cellule 2 : Vérification Inputs + GPU
# ════════════════════════════════════════════════════════════════
import json
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/EXODUS_V3/M3")
CHECKS = [
    ("avatar.glb",          DRIVE_ROOT / "SHARED"            / "avatar.glb"),
    ("decor.glb",           DRIVE_ROOT / "SHARED"            / "decor.glb"),
    ("spawn_config.json",   DRIVE_ROOT / "F03_SCENOGRAPHY"   / "OUT" / "spawn_config.json"),
    ("camera_config.json",  DRIVE_ROOT / "F04_PHOTOGRAPHY"   / "OUT" / "camera_config.json"),
    ("light_config.json",   DRIVE_ROOT / "F04_PHOTOGRAPHY"   / "OUT" / "light_config.json"),
]

ok_count = 0
for label, path in CHECKS:
    exists = path.exists()
    sz = path.stat().st_size // 1024 if exists else 0
    status = f"OK   ({sz} KB)" if exists else "ABSENT ⚠"
    print(f"{label:25s} {status}")
    if exists: ok_count += 1

print(f"\n{ok_count}/5 inputs présents")

# Vérif GPU
import subprocess
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
    print(f"GPU : {gpu}")
except Exception:
    print("GPU : non détecté — activer T4 dans les paramètres Colab")

# Aperçu spawn + trajectoire
spawn_path = DRIVE_ROOT / "F03_SCENOGRAPHY" / "OUT" / "spawn_config.json"
if spawn_path.exists():
    with open(spawn_path) as f:
        sp = json.load(f)
    traj = sp.get("trajectory", {})
    print(f"\nSpawn  : {sp.get('spawn')}")
    print(f"Scale  : {sp.get('scale')} | rot_y : {sp.get('rot_y')}")
    print(f"Traj   : mode={traj.get('mode')}  start={traj.get('start')}  end={traj.get('end')}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cellule 3 : Lancement Flask
# ════════════════════════════════════════════════════════════════
import shutil, threading, time, os
from pathlib import Path
from google.colab.output import eval_js

LOCAL = Path("/content/m3_f05")
LOCAL.mkdir(exist_ok=True)
CODEBASE = Path("/content/drive/MyDrive/EXODUS_V2/03_MODE_ASCENSION/F05_ALCHEMIST/CODEBASE")
for f in CODEBASE.glob("*"):
    shutil.copy(f, LOCAL / f.name)

PORT = 5005

def run_flask():
    os.chdir(str(LOCAL))
    os.system("python m3_f05_flask.py")

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(3)

url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print(f"\nViewer F05 ALCHEMIST :")
print(url)
print("\nOuvrir l'URL → régler PostFX → cliquer LANCER RENDER")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cellule 4 : Monitoring render + vérif frames
# ════════════════════════════════════════════════════════════════
import time, json, urllib.request
from pathlib import Path

OUT_FRAMES = Path("/content/drive/MyDrive/EXODUS_V3/M3/F05_ALCHEMIST/OUT_FRAMES")
CHECKPOINT = Path("/content/drive/MyDrive/EXODUS_V3/M3/F05_ALCHEMIST/OUT/m3_f05_checkpoint.json")

def check_status():
    try:
        with urllib.request.urlopen("http://localhost:5005/status", timeout=5) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"error": str(e)}

# Polling jusqu'à DONE / CANCELLED / ERROR
print("Monitoring render... (Ctrl+C pour arrêter)")
try:
    while True:
        s = check_status()
        if "error" in s and "status" not in s:
            print(f"Flask unreachable : {s['error']}")
            break
        status = s.get("status", "?")
        frame  = s.get("frame_current", 0)
        total  = s.get("total_frames",  0)
        pct    = s.get("pct", 0.0)
        eta    = s.get("eta_s", 0)
        bar    = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
        eta_m  = f"{eta//60}m{eta%60:02d}s" if eta > 0 else "—"
        print(f"[{bar}] {pct:5.1f}%  frame {frame:4d}/{total}  ETA {eta_m}  [{status}]", end="\r")
        if status in ("DONE", "CANCELLED", "ERROR"):
            print()
            if status == "DONE":
                n = len(list(OUT_FRAMES.glob("*.png"))) if OUT_FRAMES.exists() else 0
                print(f"Render terminé — {n} frames dans {OUT_FRAMES}")
            elif status == "ERROR":
                print(f"Erreur : {s.get('error')}")
            break
        time.sleep(2)
except KeyboardInterrupt:
    print("\nMonitoring interrompu")

# Checkpoint
if CHECKPOINT.exists():
    with open(CHECKPOINT) as f:
        ck = json.load(f)
    print(f"Checkpoint : dernière frame={ck.get('last_frame')} / {ck.get('total_frames')}")